# Model initiation

In [1]:
from faster_whisper import WhisperModel, BatchedInferencePipeline
from torch.cuda import is_available

device = 'cuda' if is_available() else 'cpu'
model_size = 'turbo'
model = WhisperModel(model_size, device=device, compute_type='int8_float16')
model = BatchedInferencePipeline(model)

# Transcribe

In [2]:
from glob import glob
from tqdm import tqdm

vid_paths = [r"D:\AIC26\vids\L25_V050.mp4"]
transcriptions = {}

for path in tqdm(vid_paths):
    segments, info = model.transcribe(
        path,
        language='vi',
        beam_size=5,
        vad_filter=True,
        batch_size=24,
        word_timestamps=True,
    )
    code = path.split('\\')[-1].split('.')[0]

    trans_words = []
    for segment in segments:
        print(f'{segment.start:.2f}', f'{segment.end:.2f}', segment.text)   
        if not segment.words: continue
        trans_words += [{'start_time': round(float(word.start), 3), 'end_time': round(float(word.end), 3), 'text': word.word.strip()} for word in segment.words]
    transcriptions[code] = trans_words

  0%|          | 0/1 [00:00<?, ?it/s]

2.48 31.17  Xin chào các bạn, mời các bạn cùng đến với các chuyên đề luyện thi tốt nghiệp trung học phổ thông quốc gia môn tiếng Anh. Và trong chuyên đề ngày hôm nay chúng ta sẽ cùng ôn tập lại các cấu trúc đạo ngữ. Thì câu hỏi về các cấu trúc đạo ngữ có vẻ như là một câu hỏi khó, tuy nhiên thì các bạn hoàn toàn có thể chinh phục được nếu các bạn nắm được các cấu trúc trong bài học ngày hôm nay. Đầu tiên là cấu trúc đạo ngữ với No Sooner.
33.03 61.05  Trong cấu trúc đạo ngữ với No Sooner chúng ta có 2 sự kiện. Thì sự kiện đằng sau No Sooner sẽ được đạo ngữ và sau đó là Dan và sau đó là một mệnh đề không được đạo ngữ. Ví dụ No Sooner had they arrived. Hôm Dan it started to rain. Câu này có nghĩa là ngay khi họ về nhà thì trời bắt đầu mưa. Tương tự nghĩa với cấu trúc No Sooner chúng ta có cấu trúc bắt đầu bằng Hartley.
61.53 91.66  Và hotly cũng có 2 mệnh đề. Mệnh đề đằng sau hotly sẽ được đảo ngữ và sau đó là when và mệnh đề thứ 2 sẽ không đảo ngữ. Và về nghĩa thì nó cũng giống với no s

100%|██████████| 1/1 [00:33<00:00, 33.27s/it]

1418.16 1446.94  We couldn't solve the problem until our teacher arrived. Câu này có nghĩa là chúng tôi đã không thể giải quyết được vấn đề cho đến khi giáo viên đến. Vậy thứ tự của hai sự kiện này là giáo viên đến thì chúng tôi mới có thể giải quyết được. Vậy chúng ta cùng xem câu A. Not until we solve the problem. Mãi cho đến khi chúng tôi giải quyết vấn đề. Could our teacher arrive thì giáo viên mới có thể đến thì sai nghĩa. Câu B.
1452.16 1477.30  Khi giáo viên đến thì chúng tôi đã giải quyết vấn đề trước đó rồi. Cũng sai nghĩa luôn. Chúng ta cùng xem câu C. Cho đến khi giáo viên đến. Chúng tôi đã có thể giải quyết được vấn đề trước đó. Cũng sai về nghĩa luôn. Và chúng ta cùng xem câu D. Cho đến khi giáo viên đến.
1477.67 1508.71  Could we solve the problem? Thì chúng tôi mới có thể giải quyết vấn đề được. Vậy câu D là câu đáp án phù hợp nhất về nghĩa và thứ tự. Câu số 10. John didn't understand what the story was about until he saw the film based on it. Câu này có nghĩa là John đã

In [30]:
trans_sentences = {}
for vid_id, words in transcriptions.items():
    sentences = []
    buffer = []
    cur_start = 0
    for word in words:
        if len(buffer) == 0: cur_start = word["start_time"]
        buffer.append(word["text"])
        if word["text"][-1] in ['.']:
            sentences.append({
                "start_time": cur_start,
                "end_time": word["end_time"],
                "length": word["end_time"] - cur_start,
                "text": ' '.join(buffer)
            })
            buffer = []
        # elif word["text"].istitle() and len(buffer) != 0:
        #     sentences.append({
        #         "start_time": cur_start,
        #         "end_time": word["end_time"],
        #         "text": ' '.join(buffer[:-1])
        #     })
        #     buffer = [word["text"]]
    trans_sentences[vid_id] = sentences

# Saving

In [1]:
import pandas as pd
print(pd.DataFrame(transcriptions["L25_V050"]).to_string())

NameError: name 'transcriptions' is not defined

In [ ]:
import json

for code, trans_word in tqdm(list(transcriptions.items())):
    parent = r'..\\additional_data\\transcript_vid'
    with open(f'{parent}\\{code}.json', 'w', encoding='utf-8') as f:
        json.dump(trans_word, f)

100%|██████████| 31/31 [00:00<00:00, 87.16it/s]
